In [ ]:
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_message_histories import ChatMessageHistory
import time
import os

# Set your Gemini Key
# os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"

def start_rua_hybrid_master():
    # --- Load Models ---
    print("👂 Powering up RUA's Ear (Faster-Whisper)...")
    stt_model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=4)
    
    print("🏠 Initializing Local Brain (Llama 3.2)...")
    local_llm = ChatOllama(model="llama3.2", temperature=0.3)
    
    print("☁️  Initializing Cloud Brain (Gemini 1.5)...")
    cloud_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3,google_api_key=os.environ.get("GOOGLE_API_KEY"))
    
    memory = ChatMessageHistory()
    recognizer = sr.Recognizer()
    recognizer.pause_threshold = 0.8
    recognizer.non_speaking_duration = 0.5

    print("\n⚡ RUA Hybrid v5: Llama + Gemini + Word-by-Word ON.")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            recognizer.adjust_for_ambient_noise(source, duration=1.0)
            
            while True:
                print("\n🟢 Listening...")
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=10)
                
                # --- 1. STT STAGE (Word-by-Word Print) ---
                stt_start = time.time()
                raw_data = audio.get_raw_data()
                audio_np = np.frombuffer(raw_data, dtype=np.int16).astype(np.float32) / 32768.0
                
                segments, info = stt_model.transcribe(audio_np, beam_size=1, word_timestamps=True)
                
                detected_lang = info.language
                print(f"👤 You ({detected_lang.upper()}): ", end="", flush=True)
                
                full_user_text = ""
                for segment in segments:
                    for word in segment.words:
                        print(word.word, end="", flush=True)
                        full_user_text += word.word
                
                stt_latency = time.time() - stt_start

                if full_user_text.strip():
                    # --- 2. HYBRID ROUTING LOGIC ---
                    # Trigger Cloud for complex tasks or explicit requests
                    complex_keywords = ['research', 'explain', 'code', 'write', 'summarize', 'plan']
                    is_complex = any(k in full_user_text.lower() for k in complex_keywords)
                    
                    if is_complex:
                        selected_llm = cloud_llm
                        brain_label = "GEMINI (Cloud)"
                    else:
                        selected_llm = local_llm
                        brain_label = "LLAMA (Local)"

                    # --- 3. DYNAMIC SYSTEM PROMPT ---
                    if detected_lang == "en":
                        lang_instruction = "Respond ONLY in English. NO Hindi."
                    else:
                        lang_instruction = "Respond in natural Hinglish (Hindi + English mix)."
                    
                    system_prompt = f"""Your name is RUA. {lang_instruction}
                    1. Be witty and concise (under 30 words).
                    2. The user is Yanil. If they ask for their name, it is Yanil.
                    3. Use history for context. No markdown/bold."""

                    # --- 4. LLM STAGE ---
                    llm_start = time.time()
                    print(f"\n🤖 RUA [{brain_label}]: ", end="", flush=True)
                    
                    messages = [{"role": "system", "content": system_prompt}]
                    # Feed last 6 messages from memory
                    for msg in memory.messages[-6:]:
                        role = "user" if msg.type == "human" else "assistant"
                        messages.append({"role": role, "content": msg.content})
                    messages.append({"role": "user", "content": full_user_text})
                    
                    full_response = ""
                    # Stream the response chunk by chunk
                    for chunk in selected_llm.stream(messages):
                        # Handle content extraction for different LLM types
                        content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                        print(content, end="", flush=True)
                        full_response += content
                    
                    llm_latency = time.time() - llm_start
                    
                    # Update History
                    memory.add_user_message(full_user_text)
                    memory.add_ai_message(full_response)
                    
                    print(f"\n\n[METRICS] STT: {stt_latency:.2f}s | LLM ({brain_label}): {llm_latency:.2f}s")
                    print("-" * 30)
                else:
                    print("\r⚪ Silence.")

    except KeyboardInterrupt:
        print("\n🛑 RUA shutting down.")

if __name__ == "__main__":
    start_rua_hybrid_master()

👂 Powering up RUA's Ear (Faster-Whisper)...
🏠 Initializing Local Brain (Llama 3.2)...
☁️  Initializing Cloud Brain (Gemini 1.5)...

⚡ RUA Hybrid v5: Llama + Gemini + Word-by-Word ON.

🟢 Listening...
👤 You (EN):  Explain Lang chain.
🤖 RUA [GEMINI (Cloud)]: 

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}